# 🚀 OmniParser Full GPU Server trên Google Colab (100% Original Pipeline)

Notebook này khởi chạy **100% Pipeline gốc của Microsoft OmniParser** (YOLOv8 Icon Detection + Florence-2 Full Semantics Captioning + EasyOCR) trên GPU miễn phí của Google Colab (Tesla T4) và xuất ra đường link công khai (`https://xxxx.trycloudflare.com`) để kết nối về máy tính Linux của bạn.

### Hướng dẫn sử dụng:
1. Trên thanh menu Colab, chọn **Runtime -> Change runtime type -> T4 GPU**.
2. Bấm **Runtime -> Run all** (hoặc chạy lần lượt 3 bước bên dưới).
3. Copy đường link `https://xxxx.trycloudflare.com` xuất hiện ở Bước 3 và dán vào lệnh chạy trên máy Linux của bạn.

In [ ]:
# @title Bước 1: Kiểm tra GPU & Cài đặt thư viện + Tải Model Weights gốc
!nvidia-smi

!git clone https://github.com/doantrongthai/fix_omni_parser.git /content/OmniParser
%cd /content/OmniParser

# Cài đặt các thư viện cần thiết cho pipeline
!pip install -q ultralytics supervision==0.18.0 easyocr transformers==4.46.3 accelerate timm einops fastapi uvicorn huggingface_hub tabulate

# Tải bộ trọng số gốc của Microsoft OmniParser v2.0 (Icon Detect + Florence-2 Caption)
from huggingface_hub import snapshot_download
import os, shutil

os.makedirs("/content/OmniParser/weights", exist_ok=True)
snapshot_download(repo_id="microsoft/OmniParser-v2.0", local_dir="/content/OmniParser/weights")
if os.path.exists("/content/OmniParser/weights/icon_caption") and not os.path.exists("/content/OmniParser/weights/icon_caption_florence"):
    shutil.move("/content/OmniParser/weights/icon_caption", "/content/OmniParser/weights/icon_caption_florence")

# Tải công cụ Cloudflare Tunnel (miễn phí, không cần tài khoản)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("✅ Đã cài đặt xong toàn bộ môi trường và trọng số gốc!")

In [ ]:
# @title Bước 3: Mở Cloudflare Tunnel & Chạy Server (Copy URL bên dưới về máy Linux)
import subprocess, threading, re, time

def start_tunnel():
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            print("\n" + "="*75)
            print(f"🌐 LINK PUBLIC COLAB GPU CỦA BẠN: {public_url}")
            print("="*75)
            print("\n👉 Copy lệnh sau chạy ở Terminal 2 trên máy Linux của bạn:")
            print(f"python app.py --windows_host_url localhost:5000 --omniparser_server_url {public_url}\n")
            print("="*75 + "\n")

# 1. Khởi chạy Cloudflare Tunnel ngầm
threading.Thread(target=start_tunnel, daemon=True).start()

# 2. Khởi chạy Uvicorn Server trong Thread riêng để tránh xung đột asyncio với Colab
def run_uvicorn():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

server_thread = threading.Thread(target=run_uvicorn, daemon=True)
server_thread.start()

print("🚀 FastAPI Server đang hoạt động tại http://127.0.0.1:8000...")
print("⏳ Đang giữ kết nối và sẵn sàng nhận ảnh (Trace logs sẽ xuất hiện bên dưới)...\n")

# Giữ cell luôn chạy để hứng live logs và tránh ngắt kết nối
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Đã dừng Server.")

In [ ]:
# @title Bước 3: Mở Cloudflare Tunnel & Chạy Server (Copy URL bên dưới về máy Linux)
import subprocess, threading, re, time

def start_tunnel():
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            print('\n' + '='*75)
            print(f'🌐 LINK PUBLIC COLAB GPU CỦA BẠN: {public_url}')
            print('='*75)
            print('\n👉 Copy lệnh sau chạy ở Terminal 3 trên máy Linux của bạn:')
            print(f'python app.py --windows_host_url localhost:5000 --omniparser_server_url {public_url}\n')
            print('='*75 + '\n')

threading.Thread(target=start_tunnel, daemon=True).start()
time.sleep(3)
uvicorn.run(app, host='127.0.0.1', port=8000)